In [2]:
import floodlight as floodlight
import pandas as pd
import numpy as np
import os
from floodlight import XY, Pitch
import floodlight.io.kinexon as knx
import floodlight.io.dfl as dfl
from floodlight.models.space import DiscreteVoronoiModel
from floodlight.models.kinetics import MetabolicPowerModel
from floodlight.models.kinematics import DistanceModel, VelocityModel, AccelerationModel
from floodlight.transforms.filter import butterworth_lowpass

from scipy.signal import find_peaks

import matplotlib.pyplot as plt

![metadataOpenData.png](attachment:metadataOpenData.png)

## Functions

In [3]:
# --- Parse timecodes to total seconds (float) ---
# df_tags format:  mm:ss:ms  (minutes can exceed 59, e.g. "95:05:00")
# df_movements format: m:ss  (e.g. "0:07", "95:05")

def parse_mmssms(t):
    """Parse 'mm:ss:ms' → total seconds (float). Minutes may exceed 59."""
    if pd.isna(t):
        return np.nan
    parts = str(t).strip().split(':')
    mm, ss, ms = int(parts[0]), int(parts[1]), int(parts[2])
    return mm * 60 + ss + ms / 1000

def parse_mmss(t):
    """Parse 'm:ss' or 'mm:ss' → total seconds (float)."""
    if pd.isna(t):
        return np.nan
    parts = str(t).strip().split(':')
    mm, ss = int(parts[0]), int(parts[1])
    return mm * 60 + ss


In [4]:
def find_movement_matches(df_tags, df_movements, window):
    """
    For each row in df_tags, search df_movements for matching movements.

    Matching criteria:
      (1) df_tags['Nummer'] == df_movements['jID']
      (2) df_tags['Team'] is a substring of df_movements['team']
          (handles name differences like "Düsseldorf" vs "Fortuna Düsseldorf")
      (3) df_tags['start_timecode_td'] falls within
          [df_movements['start_timecode_td'] - window,
           df_movements['end_timecode_td']   + window]

    Adds to df_tags (in-place on a copy):
      n_matches   : number of matching df_movements rows found
      match_idx_1 : index of the first  matching row in df_movements
      match_idx_2 : index of the second matching row (if any)
      match_idx_N : ... and so on up to the maximum found across all tags
    """
    all_matches = []

    for _, tag_row in df_tags.iterrows():
        jid       = tag_row['Nummer']
        tag_team  = tag_row['Team']
        tag_start = tag_row['start_timecode_td']

        # Skip rows with missing jersey number, team, or timecode
        if pd.isna(jid) or pd.isna(tag_team) or pd.isna(tag_start):
            all_matches.append([])
            continue

        jid_int = int(jid)

        mask = (
            (df_movements['jID'] == jid_int) &
            (df_movements['team'].str.contains(tag_team, na=False)) &
            (tag_start >= df_movements['start_timecode_td'] - window) &
            (tag_start <= df_movements['end_timecode_td']   + window)
        )
        all_matches.append(df_movements.index[mask].tolist())

    # Build result columns dynamically based on the max number of matches found
    max_matches = max((len(m) for m in all_matches), default=0)

    result = df_tags.copy()
    result['n_matches'] = [len(m) for m in all_matches]
    for i in range(max_matches):
        result[f'match_idx_{i + 1}'] = [
            m[i] if i < len(m) else pd.NA for m in all_matches
        ]

    return result


In [5]:
DATA_DIR = '/home/max/drive/data/openData2223/'
TAGS_DIR = '/home/max/drive/projects/27_deepRunsThomas/data/'
ls_match_ids = ['J03WMX', 'J03WN1', 'J03WPY', 'J03WOH', 'J03WQQ', 'J03WOY', 'J03WR9']

In [6]:
match_id = 'J03WPY'

## Read Deep Run Annotations and Process

In [7]:
df_tags = pd.read_csv(os.path.join(TAGS_DIR, match_id+'.csv'))


In [8]:
df_tags = df_tags[~df_tags['Falsch positiv '].str.contains('X', na=False)]
df_tags = df_tags.drop(
    ['Nicht im Datensatz', 'Falsch positiv ', 'Definition', 'Kommentar'], axis=1
)


In [9]:
df_tags

,xID,player,Team,Nummer,start_timecode,end_timecode
0,9.0,Shinta Appelkamp,Düsseldorf,23.0,00:07:00,00:09:00
3,16.0,"J, Castrop",Nürnberg,17.0,01:05:00,01:06:00
5,NaN,NaN,NaN,33.0,01:13:00,NaN
6,NaN,NaN,NaN,19.0,02:22:00,NaN
7,5.0,Dawid Kownacki,Düsseldorf,9.0,02:35:00,02:37:00
...,...,...,...,...,...,...
368,11.0,"C, Daferner",Nürnberg,33.0,93:19:00,93:20:00
373,NaN,NaN,Düsseldorf,9.0,94:26:00,NaN
375,8.0,"E, Wekesser",Nürnberg,13.0,95:33:00,95:34:00
376,NaN,NaN,Düsseldorf,46.0,95:55:00,NaN


## Read Player Movement Data

In [10]:
df_movements = pd.read_csv(os.path.join(TAGS_DIR, 'movements', match_id +'.csv'))


In [17]:
df_movements

,start_frame,end_frame,peak_frame,xID,player,position,team,possession,jID,x_start,...,avg_velocity_kmh,distance_m,duration_s,half,location,start_timecode,end_timecode,direction,start_timecode_td,end_timecode_td
0,0,236,109,2,Matthias Zimmermann,RV,Fortuna Düsseldorf,Home,25,0.550573,...,8.391588,22.069548,9.48,firstHalf,Home,0:00,0:09,-0.258476,0,9
1,237,281,275,2,Matthias Zimmermann,RV,Fortuna Düsseldorf,Home,25,-5.627211,...,6.200078,3.034042,1.80,firstHalf,Home,0:09,0:11,-0.151029,9,11
2,282,307,295,2,Matthias Zimmermann,RV,Fortuna Düsseldorf,Away,25,-6.131696,...,7.160262,1.989656,1.04,firstHalf,Home,0:11,0:12,0.329865,11,12
3,308,327,323,2,Matthias Zimmermann,RV,Fortuna Düsseldorf,Away,25,-5.218349,...,7.256257,1.532258,0.80,firstHalf,Home,0:12,0:13,0.308211,12,13
4,328,369,343,2,Matthias Zimmermann,RV,Fortuna Düsseldorf,Away,25,-4.569631,...,7.152316,3.260836,1.68,firstHalf,Home,0:13,0:14,0.145252,13,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46961,76681,76797,76720,19,S. Fofana,DMZ,1. FC Nürnberg,Away,3,44.550356,...,7.327407,9.482552,4.68,secondHalf,Away,96:07,96:11,-0.893926,5767,5771
46962,76798,76832,76821,19,S. Fofana,DMZ,1. FC Nürnberg,Away,3,35.672951,...,5.181222,1.957621,1.40,secondHalf,Away,96:11,96:13,-0.725268,5771,5773
46963,76833,76905,76859,19,S. Fofana,DMZ,1. FC Nürnberg,Away,3,33.790051,...,5.162518,4.132175,2.92,secondHalf,Away,96:13,96:16,-0.827267,5773,5776
46964,76906,77013,76967,19,S. Fofana,DMZ,1. FC Nürnberg,Away,3,29.735979,...,5.659309,6.733148,4.32,secondHalf,Away,96:16,96:20,-0.077626,5776,5780


## Process timedelta columns

In [11]:

df_tags['start_timecode_td'] = df_tags['start_timecode'].apply(parse_mmssms)
df_tags['end_timecode_td']   = df_tags['end_timecode'].apply(parse_mmssms)

df_movements['start_timecode_td'] = df_movements['start_timecode'].apply(parse_mmss)
df_movements['end_timecode_td']   = df_movements['end_timecode'].apply(parse_mmss)

## Filter Movement Data Based on Deep Run Criteria

In [21]:
df_filtered_movements = df_movements[
    (df_movements.possession == df_movements.location) &
    (df_movements.speed_category.isin(['jogging', 'running', 'sprinting'])) &
    (df_movements.duration_s > 1) &
    #(df_movements.x_start > -30) &
    (df_movements.direction > 0.5)  # strong forward component
    #(df_movements.x_end > df_movements.x_start)  # ensure movement is towards opponent goal
    ].copy()

In [22]:
df_filtered_movements

,start_frame,end_frame,peak_frame,xID,player,position,team,possession,jID,x_start,...,avg_velocity_kmh,distance_m,duration_s,half,location,start_timecode,end_timecode,direction,start_timecode_td,end_timecode_td
21,1319,1382,1357,2,Matthias Zimmermann,RV,Fortuna Düsseldorf,Home,25,15.878321,...,5.467266,3.881713,2.56,firstHalf,Home,0:52,0:55,0.592584,52,55
55,3510,3594,3563,2,Matthias Zimmermann,RV,Fortuna Düsseldorf,Home,25,-8.759974,...,7.161532,6.699008,3.40,firstHalf,Home,2:20,2:23,0.595576,140,143
56,3595,3680,3645,2,Matthias Zimmermann,RV,Fortuna Düsseldorf,Home,25,-3.268352,...,8.199712,7.789147,3.44,firstHalf,Home,2:23,2:27,0.916066,143,147
118,8622,8722,8664,2,Matthias Zimmermann,RV,Fortuna Düsseldorf,Home,25,-32.039718,...,6.451555,7.219809,4.04,firstHalf,Home,5:44,5:48,0.937050,344,348
120,8892,9039,8990,2,Matthias Zimmermann,RV,Fortuna Düsseldorf,Home,25,-23.078659,...,17.606847,28.797349,5.92,firstHalf,Home,5:55,6:01,0.740435,355,361
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46654,72821,73061,72910,16,J. Castrop,ORM,1. FC Nürnberg,Away,17,-7.283103,...,14.796258,39.515549,9.64,secondHalf,Away,93:32,93:42,0.766180,5612,5622
46692,76175,76399,76338,16,J. Castrop,ORM,1. FC Nürnberg,Away,17,19.351132,...,8.258832,20.631102,9.00,secondHalf,Away,95:47,95:55,0.807660,5747,5755
46753,60884,60927,60893,19,S. Fofana,DMZ,1. FC Nürnberg,Away,3,7.886056,...,2.240067,1.091927,1.76,secondHalf,Away,85:35,85:37,0.889926,5135,5137
46856,70004,70172,70062,19,S. Fofana,DMZ,1. FC Nürnberg,Away,3,17.870532,...,11.343585,21.244936,6.76,secondHalf,Away,91:40,91:46,0.649566,5500,5506


In [13]:
MATCH_WINDOW = 4  # seconds
df_tags = find_movement_matches(df_tags, df_filtered_movements, window=MATCH_WINDOW)

In [14]:

# --- Summary ---
total        = len(df_tags)
has_match    = (df_tags['n_matches'] >= 1).sum()
no_match     = (df_tags['n_matches'] == 0).sum()
multi_match  = (df_tags['n_matches'] >  1).sum()

print(f"Total tags:                   {total}")
print(f"Tags with ≥ 1 match:          {has_match}  ({has_match/total*100:.1f}%)")
print(f"Tags with 0 matches:          {no_match}  ({no_match/total*100:.1f}%)")
print(f"Tags with > 1 match:          {multi_match}  ({multi_match/total*100:.1f}%)")
print()
print("Match count distribution:")
print(df_tags['n_matches'].value_counts().sort_index())

df_tags


Total tags:                   190
Tags with ≥ 1 match:          121  (63.7%)
Tags with 0 matches:          69  (36.3%)
Tags with > 1 match:          79  (41.6%)

Match count distribution:
n_matches
0    69
1    42
2    51
3    25
4     3
Name: count, dtype: int64


,xID,player,Team,Nummer,start_timecode,end_timecode,start_timecode_td,end_timecode_td,n_matches,match_idx_1,match_idx_2,match_idx_3,match_idx_4
0,9.0,Shinta Appelkamp,Düsseldorf,23.0,00:07:00,00:09:00,7.0,9.0,2,5713,5714,<NA>,<NA>
3,16.0,"J, Castrop",Nürnberg,17.0,01:05:00,01:06:00,65.0,66.0,3,19718,19719,19720,<NA>
5,NaN,NaN,NaN,33.0,01:13:00,NaN,73.0,NaN,0,<NA>,<NA>,<NA>,<NA>
6,NaN,NaN,NaN,19.0,02:22:00,NaN,142.0,NaN,0,<NA>,<NA>,<NA>,<NA>
7,5.0,Dawid Kownacki,Düsseldorf,9.0,02:35:00,02:37:00,155.0,157.0,3,2810,2811,2812,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...
368,11.0,"C, Daferner",Nürnberg,33.0,93:19:00,93:20:00,5599.0,5600.0,2,41498,41500,<NA>,<NA>
373,NaN,NaN,Düsseldorf,9.0,94:26:00,NaN,5666.0,NaN,1,25538,<NA>,<NA>,<NA>
375,8.0,"E, Wekesser",Nürnberg,13.0,95:33:00,95:34:00,5733.0,5734.0,0,<NA>,<NA>,<NA>,<NA>
376,NaN,NaN,Düsseldorf,46.0,95:55:00,NaN,5755.0,NaN,0,<NA>,<NA>,<NA>,<NA>


In [119]:
df_filtered_movements[
    (df_filtered_movements.jID == 23) &
    (df_filtered_movements.start_timecode_td > df_tags.iloc[0]['start_timecode_td'] - 3) &
        (df_filtered_movements.start_timecode_td < df_tags.iloc[0]['start_timecode_td'] + 3)
    ]

,start_frame,end_frame,peak_frame,xID,player,position,team,possession,jID,x_start,...,avg_velocity_kmh,distance_m,duration_s,half,location,start_timecode,end_timecode,direction,start_timecode_td,end_timecode_td


In [107]:
df_movements[
    (df_movements.jID == 23) &
    (df_movements.start_timecode_td > df_tags.iloc[0]['start_timecode_td'] - 3) &
        (df_movements.start_timecode_td < df_tags.iloc[0]['start_timecode_td'] + 3)
    ]

,start_frame,end_frame,peak_frame,xID,player,position,team,possession,jID,x_start,...,avg_velocity_kmh,distance_m,duration_s,half,location,start_timecode,end_timecode,direction,start_timecode_td,end_timecode_td
5714,166,264,214,9,Shinta Appelkamp,ZO,Fortuna Düsseldorf,Home,23,-14.800011,...,18.796068,20.534335,3.96,firstHalf,Home,0:06,0:10,-0.866758,6,10
15131,235,381,362,10,Kwadwo Duah,STR,1. FC Nürnberg,Home,23,-2.740883,...,9.280613,15.073076,5.88,firstHalf,Away,0:09,0:15,0.335812,9,15


In [41]:
df_tags.sort_values('start_timecode_td')

,xID,player,Team,Nummer,start_timecode,end_timecode,start_timecode_td,end_timecode_td,n_matches,match_idx_1,match_idx_2,match_idx_3
1,16.0,"M, Karbownik",Düsseldorf,8.0,00:59:00,01:01:00,59.0,61.0,0,<NA>,<NA>,<NA>
2,10.0,Kwadwo Duah,Nürnberg,23.0,01:02:00,01:04:00,62.0,64.0,0,<NA>,<NA>,<NA>
4,2.0,Matthias Zimmermann,Düsseldorf,25.0,01:12:00,01:15:00,72.0,75.0,0,<NA>,<NA>,<NA>
9,5.0,Dawid Kownacki,Düsseldorf,9.0,02:41:00,02:42:00,161.0,162.0,0,<NA>,<NA>,<NA>
14,10.0,Kwadwo Duah,Nürnberg,23.0,03:38:00,03:40:00,218.0,220.0,0,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...
369,16.0,"M, Karbownik",Düsseldorf,8.0,93:35:00,93:36:00,5615.0,5616.0,0,<NA>,<NA>,<NA>
370,12.0,Kristoffer Peterson,Düsseldorf,7.0,94:21:00,94:25:00,5661.0,5665.0,1,30052,<NA>,<NA>
371,13.0,"C, Klarer",Düsseldorf,5.0,94:21:00,94:23:00,5661.0,5663.0,0,<NA>,<NA>,<NA>
372,8.0,"N, Gavory",Düsseldorf,34.0,94:23:00,94:25:00,5663.0,5665.0,1,28025,<NA>,<NA>
